In [5]:
%load_ext autoreload
%autoreload 2
%reset -f

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers


import os
import sys
import pandas as pd
from pathlib import Path
from datetime import datetime, date, timedelta


# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers
from locallib.pandas import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.ingester.IngesterClass import Ingester
from lib.KPIHubConnection import *
from lib.query.bank import *

from datetime import date
from datetime import timedelta


In [ ]:
class ReportSummaryIngester(Ingester):
    def __init__(self, arguments):
        super().__init__(arguments)
        self.table = KPI_ReportSummary

    def update_check(self):
        self.Logger.info(f"Processing customer: {self.customer_info['Name']}")
     # Query to get the number of different ReportId from reports from the very beggining
        query =   f"""SELECT R.Id as ReportId FROM
            Report R
        LEFT JOIN Customer C ON
            R.CustomerId = C.Id
        LEFT JOIN ReportLabel RL ON
            R.Id = RL.ReportId
        LEFT JOIN Label L ON
            RL.LabelId = L.Id
        LEFT JOIN ReportType ON
            R.ReportTypeId = ReportType.Id
        LEFT JOIN ReportArea RA ON R.Id = RA.ReportId
        LEFT JOIN ReportCompliance RC ON R.Id = RC.ReportId
        LEFT JOIN ReportAreaCovered RAC ON R.Id = RAC.ReportId
        WHERE
            R.CustomerId = '{self.customer_info['CustomerId']}' AND R.DateStarted >= '{self.starting_date}' AND L.Title = 'Final Checkbox' AND RL.IsActive = 1
            AND L.Title = 'Final Checkbox'
            AND RL.IsActive = 1
        """
        reports_list_lsdb = Query(query =query).execute(CONN_DICT[self.customer_info['DBLocation']])
        num_reports_lsdb = len(reports_list_lsdb)

        #Query the reports from the KPIHub
        query_kpi_report = f"""SELECT ReportId FROM KPI_ReportSummary WHERE CustomerId = '{self.customer_info['CustomerId']}'"""
        reports_kpi_hub = Query(query = query_kpi_report).execute(KPIHub_Conn)
        num_reports_kpi_hub = len(reports_kpi_hub)
        reports_into = reports_list_lsdb[~reports_list_lsdb['ReportId'].isin(reports_kpi_hub['ReportId'])]
        reports_deleted = reports_kpi_hub[~reports_kpi_hub['ReportId'].isin(reports_list_lsdb['ReportId'])]

        self.data['reports_into'] = reports_into
        self.data['reports_deleted'] = reports_deleted
        self.data['num_reports_lsdb'] = num_reports_lsdb
        self.data['num_reports_kpi_hub'] = num_reports_kpi_hub

        
        if (num_reports_lsdb > 0):
            self.Logger.info(f"Number of reports in LSDB: {num_reports_lsdb}")
            if(num_reports_kpi_hub == 0):
                #No reports in the KPIHub
                self.Logger.info("No reports in KPIHub, starting from the beginning")
                self.check_flag = True
            else:
                #Reports in the KPIHub
                self.Logger.info(f"Number of reports in KPIHub: {num_reports_kpi_hub}")
                if len(reports_into) > 0 or len(reports_deleted) > 0:
                    self.Logger.info(f"Number of new reports into the KPIHub: {len(reports_into)}")
                    self.Logger.info(f"Number of deleted reports in the KPIHub: {len(reports_deleted)}")
                    self.check_flag = True
                else:
                    self.Logger.info("No new reports into the KPIHub or deleted reports in the KPIHub")
                    self.check_flag = False
        else:
            self.Logger.info("No reports in LSDB")
            self.check_flag = False

    def query_data(self):
        DATAHUB_COLS = ['ReportId', 'BoundaryName', 'BoundaryType', 'BoundaryMode', 'BoundaryPlant', 'BoundarySubplant', 'BoundaryRegion', 'BoundarySubRegion']
        LSDB_COLS = [
                'ReportId',
                'CustomerId',
                'ReportName',
                'ReportDate',
                'ReportYear',
                'ReportMonth',
                'ReportWeek',
                'ReportAssetLengthKm',
                'AssetCoveredLengthKm',
                'DistributionPipeKm',
                'DistributionPipeCoveredKm',
                'ServicePipeKm',
                'ServicePipeCoveredKm',
                'ReportArea',
            ]
        if self.check_flag:
            if len(self.data['reports_into']) == self.data['num_reports_lsdb']:
                self.Logger.info(f"Updating from the beginning reports")
                query = get_reports(self.customer_info['Name'], starting_date=self.starting_date, final_checkbox = True)
                reports_lsdb = query.execute(CONN_DICT[self.customer_info['DBLocation']])
            else:
                reports_temp = self.data['reports_into'].copy()
                reports_temp.db.set_query(get_reports(self.customer_info['Name'], starting_date=self.starting_date, report_id_table = '#TempReport', final_checkbox = True))
                reports_lsdb = reports_temp.db.execute(CONN_DICT[self.customer_info['DBLocation']], source_col = 'ReportId', temp_table_name = '#TempReport')
                self.data['temp_r'] = reports_lsdb
            if self.customer_info['DBLocation'] == 'EU1' or self.customer_info['DBLocation'] == 'EU2':
                reports_lsdb.db.set_query(query_reports_view(report_table = 'temp_reports'))
                reports_datahub = reports_lsdb.db.execute(DATAHUB_Conn, source_col = 'ReportId', temp_table_name = 'temp_reports')
                # Clisify (classify) the report_summary by different periods using to_period: quarter, year, month, week
                #report_summary['ReportQuarter'] = pd.to_datetime(report_summary['ReportDate']).dt.isocalendar().quarter
                reports_lsdb['ReportYear'] = pd.to_datetime(reports_lsdb['ReportDate']).dt.year
                reports_lsdb['ReportMonth'] = pd.to_datetime(reports_lsdb['ReportDate']).dt.month
                reports_lsdb['ReportWeek'] = pd.to_datetime(reports_lsdb['ReportDate']).dt.isocalendar().week
                reports = pd.merge(reports_lsdb[LSDB_COLS], reports_datahub[DATAHUB_COLS], on = 'ReportId', how = 'left')
            else:
                reports['ReportYear'] = pd.to_datetime(reports['ReportDate']).dt.year
                reports['ReportMonth'] = pd.to_datetime(reports['ReportDate']).dt.month
                reports['ReportWeek'] = pd.to_datetime(reports['ReportDate']).dt.isocalendar().week
                reports = reports_lsdb[LSDB_COLS]
            # Add/update the LastUpdated column to the reports DataFrame as current timestamp
            reports['LastUpdated'] = datetime.now()
            self.data['output'] = reports


    def sanity_check(self):
        super().sanity_check()
        df_kpi = Query(query = f"SELECT * FROM KPI_ReportSummary WHERE CustomerId = '{self.customer_info['CustomerId']}'").execute(KPIHub_Conn)
        self.Logger.info(f"Total reports from KPI_ReportSummary: {len(df_kpi)}")



In [8]:
#Testing the class
customer_list = get_customer_list(KPIHub_Conn)
customer_info = customer_list[customer_list['Name'] == 'Avacon'].iloc[0]
print(customer_info)
ingester = ReportSummaryIngester(arguments = {'conn': KPIHub_Conn})
ingester.set_customer_info(customer_info)
ingester.update_check()
ingester.query_data()
ingester.push_data()
ingester.delete_data()
ingester.sanity_check()

CustomerId     ED346655-AAF7-B2E7-43E2-3A10E455E700
Name                                         Avacon
ShortName                                    Avacon
Active                                            1
DBLocation                                      EU1
Country                                        None
LastUpdated              2026-08-21 13:14:08.360225
Name: 0, dtype: object
